# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

In [1]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

# ============================================================
# 1. 2PL ITEM RESPONSE FUNCTION AND VISUALIZATION
# ============================================================

def p_i(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))


theta_vals = np.linspace(-6, 6, 400)

curves = [
    {"a": 0.5, "b": 0, "dash": "dash"},
    {"a": 1.5, "b": -2, "dash": "solid"},
    {"a": 1.5, "b": 0, "dash": "solid"},
    {"a": 1.5, "b": 2, "dash": "solid"}
]

fig1 = go.Figure()

for curve in curves:
    a = curve["a"]
    b = curve["b"]

    p_vals = p_i(theta_vals, a, b)

    fig1.add_trace(
        go.Scatter(
            x=theta_vals,
            y=p_vals,
            mode="lines",
            name=f"a = {a}, b = {b}",
            line=dict(dash=curve["dash"], width=2.5)
        )
    )

fig1.update_layout(
    title="Two-Parameter Logistic (2PL) Item Response Curves",
    xaxis_title="Latent Ability (theta)",
    yaxis_title="P(Y = 1 | theta)",
    xaxis=dict(range=[-6, 6]),
    yaxis=dict(range=[0, 1.05]),
    template="plotly_white",
    legend=dict(
        x=0.02,
        y=0.98
    )
)

fig1.show()


# ============================================================
# 2. SEQUENTIAL LIKELIHOOD CONTRIBUTION
# ============================================================

def likelihood_single(y, theta, a, b):
    probability = p_i(theta, a, b)
    return probability**y * (1 - probability)**(1 - y)


def likelihood_history(y_history, theta, a_values, b_values):
    likelihood = np.ones_like(theta, dtype=float)

    for k in range(len(y_history)):
        likelihood *= likelihood_single(
            y_history[k],
            theta,
            a_values[k],
            b_values[k]
        )

    return likelihood


print("--------------------------------------------------")
print("TASK 2: LIKELIHOOD")
print("--------------------------------------------------")
print("Single-response likelihood:")
print("L(y_k | theta) = p_k(theta)^y_k * [1-p_k(theta)]^(1-y_k)")
print()
print("Joint likelihood for the running history:")
print("L(y^(k) | theta) = Product[L(y_i | theta)], i = 1,...,k")
print()


# ============================================================
# 3. SEQUENTIAL BAYESIAN UPDATE ON A FIXED THETA GRID
# ============================================================

theta_grid = np.linspace(-6, 6, 2001)

prior = (
    1 / np.sqrt(2 * np.pi)
) * np.exp(
    -0.5 * theta_grid**2
)

prior = prior / np.trapezoid(prior, theta_grid)


def sequential_update(previous_posterior, y, a, b, theta_grid):
    probabilities = p_i(theta_grid, a, b)

    likelihood = (
        probabilities**y
        * (1 - probabilities)**(1 - y)
    )

    unnormalized_posterior = previous_posterior * likelihood

    normalization_constant = np.trapezoid(
        unnormalized_posterior,
        theta_grid
    )

    posterior = (
        unnormalized_posterior
        / normalization_constant
    )

    return posterior


print("--------------------------------------------------")
print("TASK 3: RECURSIVE BAYESIAN UPDATE")
print("--------------------------------------------------")
print(
    "f(theta | y^(k)) is proportional to "
    "L(y_k | theta) * f(theta | y^(k-1))"
)
print()
print(
    "The posterior is normalized numerically using the "
    "area under the posterior density."
)
print()


# ============================================================
# 4. EFFECT OF A CORRECT ANSWER TO A DIFFICULT ITEM
# ============================================================

previous_posterior = prior.copy()

a_test = 1.5
b_test = 2.0
y_test = 1

new_posterior = sequential_update(
    previous_posterior,
    y_test,
    a_test,
    b_test,
    theta_grid
)

fig2 = go.Figure()

fig2.add_trace(
    go.Scatter(
        x=theta_grid,
        y=previous_posterior,
        mode="lines",
        name="Previous Posterior",
        line=dict(width=2)
    )
)

fig2.add_trace(
    go.Scatter(
        x=theta_grid,
        y=new_posterior,
        mode="lines",
        name="After Correct Answer",
        line=dict(width=2)
    )
)

fig2.update_layout(
    title="Posterior Shift After Correct Answer to a Difficult Item",
    xaxis_title="Latent Ability (theta)",
    yaxis_title="Posterior Density",
    template="plotly_white"
)

fig2.show()


# ============================================================
# 5. EFFECT OF DISCRIMINATION PARAMETER a
# ============================================================

theta_grid_sharpness = np.linspace(-6, 6, 2001)

prior_sharpness = (
    1 / np.sqrt(2 * np.pi)
) * np.exp(
    -0.5 * theta_grid_sharpness**2
)

prior_sharpness = (
    prior_sharpness
    / np.trapezoid(
        prior_sharpness,
        theta_grid_sharpness
    )
)

b_test = 0
y_test = 1

a_values = [0.3, 0.8, 1.5, 3.0]

fig3 = go.Figure()

for a in a_values:

    posterior = sequential_update(
        prior_sharpness,
        y_test,
        a,
        b_test,
        theta_grid_sharpness
    )

    fig3.add_trace(
        go.Scatter(
            x=theta_grid_sharpness,
            y=posterior,
            mode="lines",
            name=f"a = {a}"
        )
    )

fig3.update_layout(
    title="Effect of Item Discrimination on Posterior Sharpness",
    xaxis_title="Latent Ability (theta)",
    yaxis_title="Posterior Density",
    template="plotly_white"
)

fig3.show()


# ============================================================
# 6. NUMERICAL GRID-BASED SEQUENTIAL BAYESIAN UPDATE
# ============================================================

def posterior_mean(theta_grid, posterior):
    return np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )


def posterior_variance(theta_grid, posterior):
    mean = posterior_mean(theta_grid, posterior)

    return np.trapezoid(
        (theta_grid - mean)**2 * posterior,
        theta_grid
    )


def posterior_map(theta_grid, posterior):
    index = np.argmax(posterior)
    return theta_grid[index]


print("--------------------------------------------------")
print("TASK 6: GRID-BASED NUMERICAL IMPLEMENTATION")
print("--------------------------------------------------")
print("1. Create a fixed grid of theta values.")
print("2. Initialize the grid with the N(0,1) prior.")
print("3. Observe one response at a time.")
print("4. Calculate the likelihood on every grid point.")
print("5. Multiply the previous posterior by the likelihood.")
print("6. Calculate the area using numerical integration.")
print("7. Divide by the area to normalize the posterior.")
print("8. Use the normalized posterior for the next item.")
print()
print("Normalization:")
print(
    "posterior = unnormalized_posterior / "
    "trapz(unnormalized_posterior, theta_grid)"
)
print()


# ============================================================
# 7. RUNNING SIMULATION FOR n = 20 ITEMS
# ============================================================

theta_true = 0.75
n_items = 20

theta_grid = np.linspace(-6, 6, 3001)

prior = (
    1 / np.sqrt(2 * np.pi)
) * np.exp(
    -0.5 * theta_grid**2
)

prior = prior / np.trapezoid(
    prior,
    theta_grid
)

posterior = prior.copy()

a_items = np.random.uniform(
    0.5,
    2.0,
    n_items
)

b_items = np.random.normal(
    0,
    1,
    n_items
)

responses = []

bayes_estimates = [posterior_mean(theta_grid, posterior)]
map_estimates = [posterior_map(theta_grid, posterior)]
posterior_variances = [
    posterior_variance(theta_grid, posterior)
]

steps = [0]


for k in range(n_items):

    a_k = a_items[k]
    b_k = b_items[k]

    true_probability = p_i(
        theta_true,
        a_k,
        b_k
    )

    random_number = np.random.uniform(0, 1)

    if random_number < true_probability:
        y_k = 1
    else:
        y_k = 0

    responses.append(y_k)

    posterior = sequential_update(
        posterior,
        y_k,
        a_k,
        b_k,
        theta_grid
    )

    bayes_estimate = posterior_mean(
        theta_grid,
        posterior
    )

    map_estimate = posterior_map(
        theta_grid,
        posterior
    )

    variance = posterior_variance(
        theta_grid,
        posterior
    )

    steps.append(k + 1)
    bayes_estimates.append(bayes_estimate)
    map_estimates.append(map_estimate)
    posterior_variances.append(variance)


# ============================================================
# DISPLAY SIMULATION RESULTS
# ============================================================

print("==================================================")
print("SEQUENTIAL 2PL ITEM RESPONSE SIMULATION")
print("==================================================")
print(f"True ability = {theta_true}")
print(f"Number of items = {n_items}")
print()

print(
    f"{'Step':<6}"
    f"{'a_k':<10}"
    f"{'b_k':<10}"
    f"{'Y_k':<8}"
    f"{'Bayes Mean':<15}"
    f"{'MAP':<12}"
    f"{'Variance':<12}"
)

print("-" * 75)

for k in range(n_items):

    print(
        f"{k+1:<6}"
        f"{a_items[k]:<10.3f}"
        f"{b_items[k]:<10.3f}"
        f"{responses[k]:<8}"
        f"{bayes_estimates[k+1]:<15.3f}"
        f"{map_estimates[k+1]:<12.3f}"
        f"{posterior_variances[k+1]:<12.4f}"
    )


# ============================================================
# PLOT RUNNING BAYES AND MAP ESTIMATORS
# ============================================================

fig4 = go.Figure()

fig4.add_trace(
    go.Scatter(
        x=steps,
        y=bayes_estimates,
        mode="lines+markers",
        name="Posterior Mean",
        line=dict(width=3),
        marker=dict(size=7)
    )
)

fig4.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate",
        line=dict(width=3),
        marker=dict(size=7)
    )
)

fig4.add_trace(
    go.Scatter(
        x=steps,
        y=[theta_true] * len(steps),
        mode="lines",
        name="True Ability = 0.75",
        line=dict(
            dash="dash",
            width=3
        )
    )
)

fig4.update_layout(
    title="Running Ability Estimation Using Sequential Bayesian Updating",
    xaxis_title="Number of Items Answered (k)",
    yaxis_title="Estimated Ability (theta)",
    xaxis=dict(
        dtick=1
    ),
    template="plotly_white",
    legend=dict(
        x=0.02,
        y=0.98
    ),
    hovermode="x unified"
)

fig4.show()


# ============================================================
# POSTERIOR VARIANCE / CERTAINTY PLOT
# ============================================================

fig5 = go.Figure()

fig5.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_variances,
        mode="lines+markers",
        name="Posterior Variance",
        line=dict(width=3),
        marker=dict(size=7)
    )
)

fig5.update_layout(
    title="Posterior Variance During Sequential Learning",
    xaxis_title="Number of Items Answered (k)",
    yaxis_title="Posterior Variance",
    xaxis=dict(
        dtick=1
    ),
    template="plotly_white"
)

fig5.show()


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("==================================================")
print("FINAL ESTIMATION SUMMARY")
print("==================================================")

print(f"True ability:              {theta_true:.4f}")
print(f"Final Posterior Mean:      {bayes_estimates[-1]:.4f}")
print(f"Final MAP estimate:        {map_estimates[-1]:.4f}")
print(f"Final Posterior Variance:  {posterior_variances[-1]:.4f}")

print()
print(
    "The posterior mean and MAP estimates are updated after "
    "every response using the new item's likelihood."
)

print(
    "As more informative responses are observed, the estimates "
    "generally move toward the true ability."
)

print(
    "A decreasing posterior variance indicates increasing "
    "certainty about the user's latent ability."
)

--------------------------------------------------
TASK 2: LIKELIHOOD
--------------------------------------------------
Single-response likelihood:
L(y_k | theta) = p_k(theta)^y_k * [1-p_k(theta)]^(1-y_k)

Joint likelihood for the running history:
L(y^(k) | theta) = Product[L(y_i | theta)], i = 1,...,k

--------------------------------------------------
TASK 3: RECURSIVE BAYESIAN UPDATE
--------------------------------------------------
f(theta | y^(k)) is proportional to L(y_k | theta) * f(theta | y^(k-1))

The posterior is normalized numerically using the area under the posterior density.



--------------------------------------------------
TASK 6: GRID-BASED NUMERICAL IMPLEMENTATION
--------------------------------------------------
1. Create a fixed grid of theta values.
2. Initialize the grid with the N(0,1) prior.
3. Observe one response at a time.
4. Calculate the likelihood on every grid point.
5. Multiply the previous posterior by the likelihood.
6. Calculate the area using numerical integration.
7. Divide by the area to normalize the posterior.
8. Use the normalized posterior for the next item.

Normalization:
posterior = unnormalized_posterior / trapz(unnormalized_posterior, theta_grid)

SEQUENTIAL 2PL ITEM RESPONSE SIMULATION
True ability = 0.75
Number of items = 20

Step  a_k       b_k       Y_k     Bayes Mean     MAP         Variance    
---------------------------------------------------------------------------
1     1.062     -1.013    1       0.258          0.224       0.8499      
2     1.926     0.314     1       0.809          0.736       0.5827      
3 


FINAL ESTIMATION SUMMARY
True ability:              0.7500
Final Posterior Mean:      0.8423
Final MAP estimate:        0.7920
Final Posterior Variance:  0.1964

The posterior mean and MAP estimates are updated after every response using the new item's likelihood.
As more informative responses are observed, the estimates generally move toward the true ability.
A decreasing posterior variance indicates increasing certainty about the user's latent ability.


# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

In [2]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(42)

# ============================================================
# 1. VISUALIZATION OF THE BETA DISTRIBUTION
# ============================================================

theta_grid = np.linspace(0, 1, 500)

beta_configs = [
    {
        "alpha": 1,
        "beta": 1,
        "name": "Uninformative State: Beta(1,1)",
        "dash": "dash"
    },
    {
        "alpha": 2,
        "beta": 8,
        "name": "Right-Skewed State: Beta(2,8)",
        "dash": "solid"
    },
    {
        "alpha": 8,
        "beta": 2,
        "name": "Left-Skewed State: Beta(8,2)",
        "dash": "solid"
    }
]

fig1 = go.Figure()

for config in beta_configs:

    alpha = config["alpha"]
    beta = config["beta"]

    pdf_vals = stats.beta.pdf(
        theta_grid,
        alpha,
        beta
    )

    fig1.add_trace(
        go.Scatter(
            x=theta_grid,
            y=pdf_vals,
            mode="lines",
            name=config["name"],
            line=dict(
                dash=config["dash"],
                width=2.5
            )
        )
    )

fig1.update_layout(
    title="Structural Variations of the Beta(α, β) Probability Density Function",
    xaxis_title="Click Probability (θ)",
    yaxis_title="Probability Density f(θ)",
    xaxis=dict(
        range=[0, 1]
    ),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        x=0.02,
        y=0.98
    )
)

fig1.show()


# ============================================================
# 2. SINGLE RESPONSE LIKELIHOOD AND JOINT LIKELIHOOD
# ============================================================

def single_likelihood(y, theta):
    return theta**y * (1 - theta)**(1 - y)


def joint_likelihood(responses, theta):
    responses = np.asarray(responses)

    clicks = np.sum(responses)
    non_clicks = len(responses) - clicks

    return theta**clicks * (1 - theta)**non_clicks


print("=" * 70)
print("TASK 2: LIKELIHOOD FUNCTIONS")
print("=" * 70)

print()
print("Single response likelihood:")
print("L(y_k | theta) = theta^y_k * (1 - theta)^(1-y_k)")

print()
print("For y_k = 1:")
print("L(1 | theta) = theta")

print()
print("For y_k = 0:")
print("L(0 | theta) = 1 - theta")

print()
print("Joint likelihood:")
print(
    "L(y^(k) | theta) = "
    "theta^(sum(y_i)) * "
    "(1-theta)^(k-sum(y_i))"
)

print()


# ============================================================
# 3. BETA-BINOMIAL CONJUGATE UPDATE
# ============================================================

alpha_0 = 1
beta_0 = 1

print("=" * 70)
print("TASK 3: CLOSED-FORM BETA-BINOMIAL UPDATE")
print("=" * 70)

print()
print("Recursive Bayesian relationship:")
print(
    "f(theta | y^(k)) ∝ "
    "L(y_k | theta) * f(theta | y^(k-1))"
)

print()
print("Closed-form parameter updates:")
print("alpha_k = alpha_(k-1) + y_k")
print("beta_k  = beta_(k-1) + (1-y_k)")

print()
print("Therefore:")
print("alpha_k = alpha_0 + number of clicks")
print("beta_k  = beta_0 + number of non-clicks")

print()
print("Posterior distribution:")
print(
    "Theta | y^(k) ~ Beta(alpha_k, beta_k)"
)

print()
print("Posterior Mean:")
print(
    "E[Theta | y^(k)] = "
    "alpha_k / (alpha_k + beta_k)"
)

print()
print("Expanded Posterior Mean:")
print(
    "E[Theta | y^(k)] = "
    "(alpha_0 + C_k) / "
    "(alpha_0 + beta_0 + k)"
)

print()


# ============================================================
# 4. DYNAMIC SHIFTING AFTER A CLICK AND NON-CLICK
# ============================================================

theta_plot = np.linspace(
    0,
    1,
    500
)

alpha_previous = 5
beta_previous = 5

prior_density = stats.beta.pdf(
    theta_plot,
    alpha_previous,
    beta_previous
)

alpha_after_click = alpha_previous + 1
beta_after_click = beta_previous

click_density = stats.beta.pdf(
    theta_plot,
    alpha_after_click,
    beta_after_click
)

alpha_after_no_click = alpha_previous
beta_after_no_click = beta_previous + 1

no_click_density = stats.beta.pdf(
    theta_plot,
    alpha_after_no_click,
    beta_after_no_click
)

fig2 = go.Figure()

fig2.add_trace(
    go.Scatter(
        x=theta_plot,
        y=prior_density,
        mode="lines",
        name="Previous: Beta(5,5)",
        line=dict(
            width=2.5,
            dash="dash"
        )
    )
)

fig2.add_trace(
    go.Scatter(
        x=theta_plot,
        y=click_density,
        mode="lines",
        name="After Click: Beta(6,5)",
        line=dict(
            width=2.5
        )
    )
)

fig2.add_trace(
    go.Scatter(
        x=theta_plot,
        y=no_click_density,
        mode="lines",
        name="After No Click: Beta(5,6)",
        line=dict(
            width=2.5
        )
    )
)

fig2.update_layout(
    title="Dynamic Posterior Shifting After Individual User Responses",
    xaxis_title="Click Probability (θ)",
    yaxis_title="Posterior Density",
    template="plotly_white",
    hovermode="x unified"
)

fig2.show()


# ============================================================
# 5. POSTERIOR MEAN AND MAP ESTIMATORS
# ============================================================

def posterior_mean(alpha, beta):
    return alpha / (alpha + beta)


def posterior_map(alpha, beta):

    if alpha > 1 and beta > 1:
        return (alpha - 1) / (alpha + beta - 2)

    elif alpha <= 1 and beta > 1:
        return 0.0

    elif alpha > 1 and beta <= 1:
        return 1.0

    else:
        return 0.0


print("=" * 70)
print("TASK 5: CLOSED-FORM POINT ESTIMATORS")
print("=" * 70)

print()
print("Posterior Mean:")
print(
    "theta_Bayes^(k) = "
    "alpha_k / (alpha_k + beta_k)"
)

print()
print("MAP estimate when alpha_k > 1 and beta_k > 1:")
print(
    "theta_MAP^(k) = "
    "(alpha_k - 1) / "
    "(alpha_k + beta_k - 2)"
)

print()
print("Boundary cases:")
print("If alpha_k <= 1 and beta_k > 1, MAP = 0")
print("If alpha_k > 1 and beta_k <= 1, MAP = 1")
print("If alpha_k <= 1 and beta_k <= 1, the mode is at a boundary.")

print()


# ============================================================
# 6. SEQUENTIAL CTR SIMULATION
# ============================================================

theta_true = 0.35
n_impressions = 100

alpha_param = 1
beta_param = 1

steps = list(
    range(n_impressions + 1)
)

running_bayes = [
    posterior_mean(
        alpha_param,
        beta_param
    )
]

running_map = [
    posterior_map(
        alpha_param,
        beta_param
    )
]

click_history = []

alpha_history = [alpha_param]
beta_history = [beta_param]


for k in range(
    1,
    n_impressions + 1
):

    random_number = np.random.uniform(
        0,
        1
    )

    if random_number < theta_true:
        y_k = 1
    else:
        y_k = 0

    click_history.append(y_k)

    alpha_param = (
        alpha_param + y_k
    )

    beta_param = (
        beta_param + (1 - y_k)
    )

    theta_bayes_k = posterior_mean(
        alpha_param,
        beta_param
    )

    theta_map_k = posterior_map(
        alpha_param,
        beta_param
    )

    running_bayes.append(
        theta_bayes_k
    )

    running_map.append(
        theta_map_k
    )

    alpha_history.append(
        alpha_param
    )

    beta_history.append(
        beta_param
    )


# ============================================================
# DISPLAY SIMULATION RESULTS
# ============================================================

print("=" * 90)
print("SEQUENTIAL BETA-BINOMIAL CTR ESTIMATION")
print("=" * 90)

print(
    f"{'Step':<8}"
    f"{'Y_k':<8}"
    f"{'Clicks':<10}"
    f"{'Alpha':<10}"
    f"{'Beta':<10}"
    f"{'Bayes Mean':<15}"
    f"{'MAP':<15}"
)

print("-" * 90)

total_clicks = 0

for k in range(
    n_impressions
):

    total_clicks += click_history[k]

    print(
        f"{k+1:<8}"
        f"{click_history[k]:<8}"
        f"{total_clicks:<10}"
        f"{alpha_history[k+1]:<10}"
        f"{beta_history[k+1]:<10}"
        f"{running_bayes[k+1]:<15.4f}"
        f"{running_map[k+1]:<15.4f}"
    )


# ============================================================
# FINAL VALUES
# ============================================================

final_clicks = sum(
    click_history
)

final_non_clicks = (
    n_impressions - final_clicks
)

final_alpha = alpha_history[-1]
final_beta = beta_history[-1]

final_bayes = running_bayes[-1]
final_map = running_map[-1]

print()
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    f"True CTR:                 {theta_true:.4f}"
)

print(
    f"Total impressions:        {n_impressions}"
)

print(
    f"Total clicks:             {final_clicks}"
)

print(
    f"Total non-clicks:         {final_non_clicks}"
)

print(
    f"Final alpha:              {final_alpha}"
)

print(
    f"Final beta:               {final_beta}"
)

print(
    f"Final Posterior Mean:     {final_bayes:.4f}"
)

print(
    f"Final MAP Estimate:       {final_map:.4f}"
)

print(
    f"Mean absolute distance:   "
    f"{abs(final_bayes - theta_true):.4f}"
)

print(
    f"MAP absolute distance:    "
    f"{abs(final_map - theta_true):.4f}"
)


# ============================================================
# PLOT: RUNNING ESTIMATORS FROM 0 TO 100
# ============================================================

fig3 = go.Figure()

fig3.add_trace(
    go.Scatter(
        x=steps,
        y=running_bayes,
        mode="lines+markers",
        name="Posterior Mean",
        line=dict(
            width=2.5
        ),
        marker=dict(
            size=5
        )
    )
)

fig3.add_trace(
    go.Scatter(
        x=steps,
        y=running_map,
        mode="lines+markers",
        name="MAP Estimate",
        line=dict(
            width=2.5,
            dash="dot"
        ),
        marker=dict(
            size=5
        )
    )
)

fig3.add_trace(
    go.Scatter(
        x=steps,
        y=[theta_true] * len(steps),
        mode="lines",
        name="True CTR = 0.35",
        line=dict(
            dash="dash",
            width=3
        )
    )
)

fig3.update_layout(
    title="Sequential Bayesian CTR Estimation Using Beta-Binomial Conjugacy",
    xaxis_title="Number of Impressions (k)",
    yaxis_title="Estimated CTR",
    xaxis=dict(
        range=[0, 100],
        dtick=10
    ),
    yaxis=dict(
        range=[0, 1]
    ),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        x=0.02,
        y=0.98
    )
)

fig3.show()


# ============================================================
# OPTIONAL: POSTERIOR DISTRIBUTION AT SELECTED STEPS
# ============================================================

milestones = [
    0,
    1,
    5,
    10,
    25,
    50,
    75,
    100
]

theta_pdf_grid = np.linspace(
    0,
    1,
    500
)

fig4 = go.Figure()

for milestone in milestones:

    alpha_m = alpha_history[milestone]
    beta_m = beta_history[milestone]

    density = stats.beta.pdf(
        theta_pdf_grid,
        alpha_m,
        beta_m
    )

    fig4.add_trace(
        go.Scatter(
            x=theta_pdf_grid,
            y=density,
            mode="lines",
            name=(
                f"Step {milestone}: "
                f"Beta({alpha_m},{beta_m})"
            )
        )
    )

fig4.add_vline(
    x=theta_true,
    line_dash="dash",
    line_width=2,
    annotation_text="True CTR = 0.35",
    annotation_position="top right"
)

fig4.update_layout(
    title="Evolution of the Posterior CTR Distribution",
    xaxis_title="CTR Parameter (θ)",
    yaxis_title="Posterior Density",
    template="plotly_white",
    hovermode="x unified"
)

fig4.show()


# ============================================================
# FINAL INTERPRETATION
# ============================================================

print()
print("=" * 70)
print("INTERPRETATION")
print("=" * 70)

print(
    "The initial Beta(1,1) prior represents uniform uncertainty "
    "about the CTR."
)

print(
    "Every click increases alpha by 1, while every non-click "
    "increases beta by 1."
)

print(
    "The posterior mean therefore continuously incorporates "
    "the accumulated evidence."
)

print(
    "As the number of impressions increases, the influence of "
    "the initial prior becomes progressively smaller."
)

print(
    "For sufficiently large k, the posterior mean approaches "
    "the empirical click proportion and, under repeated sampling, "
    "tends toward the true CTR."
)

print(
    "The MAP estimate also becomes increasingly stable as more "
    "observations accumulate."
)

TASK 2: LIKELIHOOD FUNCTIONS

Single response likelihood:
L(y_k | theta) = theta^y_k * (1 - theta)^(1-y_k)

For y_k = 1:
L(1 | theta) = theta

For y_k = 0:
L(0 | theta) = 1 - theta

Joint likelihood:
L(y^(k) | theta) = theta^(sum(y_i)) * (1-theta)^(k-sum(y_i))

TASK 3: CLOSED-FORM BETA-BINOMIAL UPDATE

Recursive Bayesian relationship:
f(theta | y^(k)) ∝ L(y_k | theta) * f(theta | y^(k-1))

Closed-form parameter updates:
alpha_k = alpha_(k-1) + y_k
beta_k  = beta_(k-1) + (1-y_k)

Therefore:
alpha_k = alpha_0 + number of clicks
beta_k  = beta_0 + number of non-clicks

Posterior distribution:
Theta | y^(k) ~ Beta(alpha_k, beta_k)

Posterior Mean:
E[Theta | y^(k)] = alpha_k / (alpha_k + beta_k)

Expanded Posterior Mean:
E[Theta | y^(k)] = (alpha_0 + C_k) / (alpha_0 + beta_0 + k)



TASK 5: CLOSED-FORM POINT ESTIMATORS

Posterior Mean:
theta_Bayes^(k) = alpha_k / (alpha_k + beta_k)

MAP estimate when alpha_k > 1 and beta_k > 1:
theta_MAP^(k) = (alpha_k - 1) / (alpha_k + beta_k - 2)

Boundary cases:
If alpha_k <= 1 and beta_k > 1, MAP = 0
If alpha_k > 1 and beta_k <= 1, MAP = 1
If alpha_k <= 1 and beta_k <= 1, the mode is at a boundary.

SEQUENTIAL BETA-BINOMIAL CTR ESTIMATION
Step    Y_k     Clicks    Alpha     Beta      Bayes Mean     MAP            
------------------------------------------------------------------------------------------
1       0       0         1         2         0.3333         0.0000         
2       0       0         1         3         0.2500         0.0000         
3       0       0         1         4         0.2000         0.0000         
4       0       0         1         5         0.1667         0.0000         
5       1       1         2         5         0.2857         0.2000         
6       1       2         3         5         


INTERPRETATION
The initial Beta(1,1) prior represents uniform uncertainty about the CTR.
Every click increases alpha by 1, while every non-click increases beta by 1.
The posterior mean therefore continuously incorporates the accumulated evidence.
As the number of impressions increases, the influence of the initial prior becomes progressively smaller.
For sufficiently large k, the posterior mean approaches the empirical click proportion and, under repeated sampling, tends toward the true CTR.
The MAP estimate also becomes increasingly stable as more observations accumulate.


# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In [3]:
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

np.random.seed(24)

# ============================================================
# TASK 1: PRIOR BELIEF BOUNDARIES
# ============================================================

theta_grid = np.linspace(0.01, 1.0, 1000)

alpha_prior = 8
beta_prior = 1.5

prior_density = stats.beta.pdf(
    theta_grid,
    alpha_prior,
    beta_prior
)

prior_density = (
    prior_density /
    np.trapezoid(prior_density, theta_grid)
)

prior_mean = alpha_prior / (
    alpha_prior + beta_prior
)

print("=" * 75)
print("TASK 1: PRIOR BELIEF")
print("=" * 75)

print(f"Prior distribution: Beta({alpha_prior}, {beta_prior})")
print(f"Expected prior stiffness efficiency: {prior_mean:.4f}")
print()

fig1 = go.Figure()

fig1.add_trace(
    go.Scatter(
        x=theta_grid,
        y=prior_density,
        mode="lines",
        name="Prior: Beta(8, 1.5)",
        line=dict(
            width=3,
            dash="solid"
        )
    )
)

fig1.add_vline(
    x=prior_mean,
    line_dash="dash",
    line_width=2,
    annotation_text=f"Prior Mean = {prior_mean:.3f}",
    annotation_position="top left"
)

fig1.update_layout(
    title="Initial Prior Distribution for Structural Stiffness Efficiency",
    xaxis_title="Remaining Stiffness Efficiency Factor (θ)",
    yaxis_title="Prior Probability Density",
    xaxis=dict(
        range=[0.01, 1.0]
    ),
    template="plotly_white",
    hovermode="x unified"
)

fig1.show()


# ============================================================
# TASK 2: LOG-NORMAL LIKELIHOOD
# ============================================================

def lognormal_likelihood(y, theta, K_nominal, sigma):

    expected_stiffness = theta * K_nominal

    likelihood = stats.lognorm.pdf(
        y,
        s=sigma,
        scale=expected_stiffness
    )

    return likelihood


print("=" * 75)
print("TASK 2: STRUCTURAL LIKELIHOOD")
print("=" * 75)

print()
print("Measurement model:")
print("Y_k = theta * K_nominal * exp(epsilon_k)")
print("epsilon_k ~ N(0, sigma^2)")
print()

print("Single-measurement likelihood:")
print(
    "L(y_k | theta) = "
    "1 / (y_k * sigma * sqrt(2*pi)) "
    "* exp(-(ln(y_k/(theta*K_nominal)))^2/(2*sigma^2))"
)

print()
print("Joint likelihood:")
print(
    "L(y^(k) | theta) = "
    "Product[L(y_i | theta)], i = 1,...,k"
)

print()


# ============================================================
# TASK 3: NON-CONJUGATE RECURSIVE BAYESIAN UPDATE
# ============================================================

print("=" * 75)
print("TASK 3: NON-CONJUGATE BAYESIAN UPDATE")
print("=" * 75)

print()
print(
    "The Beta prior is not conjugate to the log-normal "
    "measurement likelihood."
)

print()
print(
    "Therefore, the posterior generally does not have a "
    "closed-form standard distribution."
)

print()
print(
    "Recursive update:"
)

print(
    "f(theta | y^(k)) proportional to "
    "L(y_k | theta) * f(theta | y^(k-1))"
)

print()
print(
    "The normalized posterior is:"
)

print(
    "f(theta | y^(k)) = "
    "[L(y_k | theta) f(theta | y^(k-1))] / Z_k"
)

print()
print(
    "where Z_k is the integral of the numerator over "
    "the physical domain."
)

print()


# ============================================================
# TASK 4: RUNNING POINT ESTIMATORS
# ============================================================

def posterior_mean(theta_grid, posterior):

    return np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )


def posterior_map(theta_grid, posterior):

    index = np.argmax(posterior)

    return theta_grid[index]


print("=" * 75)
print("TASK 4: RUNNING POINT ESTIMATORS")
print("=" * 75)

print()
print(
    "Posterior Mean:"
)

print(
    "theta_Bayes = integral(theta * posterior(theta) dtheta)"
)

print()
print(
    "MAP:"
)

print(
    "theta_MAP = argmax_theta posterior(theta)"
)

print()
print(
    "Both quantities are evaluated numerically on the "
    "bounded theta grid."
)

print()


# ============================================================
# TASK 5: GRID-BASED BAYESIAN UPDATE
# ============================================================

print("=" * 75)
print("TASK 5: GRID-BASED NUMERICAL PROCEDURE")
print("=" * 75)

print()
print("Physical lower boundary: theta_min = 0.01")
print("Physical upper boundary: theta_max = 1.00")
print()

print("Sequential computational procedure:")
print("1. Create a grid over [0.01, 1.00].")
print("2. Evaluate and normalize the Beta prior.")
print("3. Receive a new stiffness measurement.")
print("4. Evaluate the log-normal likelihood at every grid point.")
print("5. Multiply the previous posterior by the likelihood.")
print("6. Compute the normalization constant using np.trapezoid.")
print("7. Divide by the normalization constant.")
print("8. Calculate posterior mean and MAP.")
print("9. Use this posterior as the prior for the next measurement.")

print()
print(
    "Normalization:"
)

print(
    "Z_k = np.trapezoid(unnormalized_posterior, theta_grid)"
)

print()

print(
    "normalized_posterior = "
    "unnormalized_posterior / Z_k"
)

print()


# ============================================================
# TASK 6: STRUCTURAL HEALTH MONITORING SIMULATION
# ============================================================

theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_sensor_readings = 15

theta_grid = np.linspace(
    0.01,
    1.0,
    2000
)

# ------------------------------------------------------------
# INITIAL PRIOR
# ------------------------------------------------------------

current_posterior = stats.beta.pdf(
    theta_grid,
    a=8,
    b=1.5
)

current_posterior = (
    current_posterior /
    np.trapezoid(
        current_posterior,
        theta_grid
    )
)

# ------------------------------------------------------------
# MILESTONES
# ------------------------------------------------------------

milestones = [
    0,
    1,
    2,
    5,
    10,
    15
]

# ------------------------------------------------------------
# TRACKING ARRAYS
# ------------------------------------------------------------

steps = [0]

bayes_estimates = [
    posterior_mean(
        theta_grid,
        current_posterior
    )
]

map_estimates = [
    posterior_map(
        theta_grid,
        current_posterior
    )
]

posterior_variances = []

initial_mean = bayes_estimates[0]

initial_variance = np.trapezoid(
    (
        theta_grid -
        initial_mean
    )**2 *
    current_posterior,
    theta_grid
)

posterior_variances.append(
    initial_variance
)

sensor_readings = []

alpha_history = []
beta_history = []

# ------------------------------------------------------------
# FIGURE FOR POSTERIOR DENSITY EVOLUTION
# ------------------------------------------------------------

fig2 = go.Figure()

fig2.add_trace(
    go.Scatter(
        x=theta_grid,
        y=current_posterior,
        mode="lines",
        name="Step 0: Prior Beta(8,1.5)",
        line=dict(
            width=3,
            dash="dash"
        )
    )
)

# ------------------------------------------------------------
# SEQUENTIAL MONITORING LOOP
# ------------------------------------------------------------

for k in range(
    1,
    n_sensor_readings + 1
):

    # Generate log-normal sensor noise
    epsilon_k = np.random.normal(
        0,
        sigma
    )

    # Generate sensor measurement
    y_k = (
        theta_true *
        K_nominal *
        np.exp(epsilon_k)
    )

    sensor_readings.append(
        y_k
    )

    # Expected stiffness at every grid point
    expected_stiffness = (
        theta_grid *
        K_nominal
    )

    # Log-normal likelihood
    likelihood = stats.lognorm.pdf(
        y_k,
        s=sigma,
        scale=expected_stiffness
    )

    # Prevent numerical problems
    likelihood = np.maximum(
        likelihood,
        1e-300
    )

    # --------------------------------------------------------
    # BAYESIAN UPDATE
    # --------------------------------------------------------

    unnormalized_posterior = (
        current_posterior *
        likelihood
    )

    # --------------------------------------------------------
    # NUMERICAL NORMALIZATION
    # --------------------------------------------------------

    normalization_constant = np.trapezoid(
        unnormalized_posterior,
        theta_grid
    )

    current_posterior = (
        unnormalized_posterior /
        normalization_constant
    )

    # --------------------------------------------------------
    # POINT ESTIMATES
    # --------------------------------------------------------

    bayes_estimate = posterior_mean(
        theta_grid,
        current_posterior
    )

    map_estimate = posterior_map(
        theta_grid,
        current_posterior
    )

    # --------------------------------------------------------
    # POSTERIOR VARIANCE
    # --------------------------------------------------------

    variance = np.trapezoid(
        (
            theta_grid -
            bayes_estimate
        )**2 *
        current_posterior,
        theta_grid
    )

    # --------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------

    steps.append(k)

    bayes_estimates.append(
        bayes_estimate
    )

    map_estimates.append(
        map_estimate
    )

    posterior_variances.append(
        variance
    )

    # --------------------------------------------------------
    # STORE MILESTONE POSTERIOR
    # --------------------------------------------------------

    if k in milestones:

        fig2.add_trace(
            go.Scatter(
                x=theta_grid,
                y=current_posterior,
                mode="lines",
                name=(
                    f"Step {k}: "
                    f"y = {y_k:.2f}"
                ),
                line=dict(
                    width=2
                )
            )
        )


# ============================================================
# POSTERIOR DENSITY MILESTONE PLOT
# ============================================================

fig2.add_vline(
    x=theta_true,
    line_dash="dot",
    line_width=3,
    annotation_text=(
        f"True Stiffness = {theta_true}"
    ),
    annotation_position="top left"
)

fig2.update_layout(
    title=(
        "Structural Health Monitoring: "
        "Sequential Posterior Evolution"
    ),
    xaxis_title=(
        "Remaining Stiffness Efficiency Factor (θ)"
    ),
    yaxis_title=(
        "Posterior Probability Density"
    ),
    xaxis=dict(
        range=[0.01, 1.0]
    ),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        x=0.70,
        y=0.98
    )
)

fig2.show()


# ============================================================
# CONVERGENCE TIMELINE
# ============================================================

fig3 = go.Figure()

fig3.add_trace(
    go.Scatter(
        x=steps,
        y=bayes_estimates,
        mode="lines+markers",
        name="Posterior Mean",
        line=dict(
            width=3
        ),
        marker=dict(
            size=7
        )
    )
)

fig3.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode="lines+markers",
        name="MAP Estimate",
        line=dict(
            width=3,
            dash="dot"
        ),
        marker=dict(
            size=7
        )
    )
)

fig3.add_trace(
    go.Scatter(
        x=steps,
        y=[
            theta_true
        ] * len(steps),
        mode="lines",
        name="True Stiffness = 0.68",
        line=dict(
            width=3,
            dash="dash"
        )
    )
)

fig3.update_layout(
    title=(
        "Sequential Structural Stiffness "
        "Estimation and Convergence"
    ),
    xaxis_title=(
        "Inspection Step (k)"
    ),
    yaxis_title=(
        "Estimated Stiffness Efficiency (θ)"
    ),
    xaxis=dict(
        range=[0, 15],
        dtick=1
    ),
    yaxis=dict(
        range=[0, 1.0]
    ),
    template="plotly_white",
    hovermode="x unified",
    legend=dict(
        x=0.02,
        y=0.98
    )
)

fig3.show()


# ============================================================
# POSTERIOR VARIANCE PLOT
# ============================================================

fig4 = go.Figure()

fig4.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_variances,
        mode="lines+markers",
        name="Posterior Variance",
        line=dict(
            width=3
        ),
        marker=dict(
            size=7
        )
    )
)

fig4.update_layout(
    title=(
        "Reduction of Posterior Uncertainty "
        "During Structural Monitoring"
    ),
    xaxis_title="Inspection Step (k)",
    yaxis_title="Posterior Variance",
    xaxis=dict(
        dtick=1
    ),
    template="plotly_white"
)

fig4.show()


# ============================================================
# RESULTS TABLE
# ============================================================

print()
print("=" * 100)
print("TASK 6: SEQUENTIAL STRUCTURAL HEALTH MONITORING RESULTS")
print("=" * 100)

print(
    f"{'Step':<8}"
    f"{'Sensor Reading':<18}"
    f"{'Bayes Mean':<18}"
    f"{'MAP':<18}"
    f"{'Variance':<18}"
    f"{'Error Mean':<15}"
)

print("-" * 100)

print(
    f"{0:<8}"
    f"{'Prior':<18}"
    f"{bayes_estimates[0]:<18.4f}"
    f"{map_estimates[0]:<18.4f}"
    f"{posterior_variances[0]:<18.6f}"
    f"{abs(bayes_estimates[0] - theta_true):<15.4f}"
)

for k in range(
    1,
    n_sensor_readings + 1
):

    print(
        f"{k:<8}"
        f"{sensor_readings[k-1]:<18.4f}"
        f"{bayes_estimates[k]:<18.4f}"
        f"{map_estimates[k]:<18.4f}"
        f"{posterior_variances[k]:<18.6f}"
        f"{abs(bayes_estimates[k] - theta_true):<15.4f}"
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print("=" * 75)
print("FINAL STRUCTURAL HEALTH SUMMARY")
print("=" * 75)

print(
    f"True remaining stiffness:       {theta_true:.4f}"
)

print(
    f"Initial prior mean:              {initial_mean:.4f}"
)

print(
    f"Final posterior mean:            "
    f"{bayes_estimates[-1]:.4f}"
)

print(
    f"Final MAP estimate:              "
    f"{map_estimates[-1]:.4f}"
)

print(
    f"Final posterior variance:        "
    f"{posterior_variances[-1]:.6f}"
)

print(
    f"Final mean absolute error:       "
    f"{abs(bayes_estimates[-1] - theta_true):.4f}"
)

print(
    f"Initial mean absolute error:     "
    f"{abs(initial_mean - theta_true):.4f}"
)

print()

print(
    "The initial Beta(8,1.5) prior strongly favors a healthy "
    "structure with theta close to 1."
)

print(
    "As sensor measurements arrive, the likelihood progressively "
    "overcomes this optimistic prior and moves the posterior "
    "toward the true degraded state."
)

print(
    "The posterior becomes narrower as repeated measurements "
    "accumulate, indicating increasing confidence in the estimated "
    "structural stiffness."
)

print(
    "The exact number of measurements required to overcome the "
    "prior depends on the random sensor sequence and noise."
)

TASK 1: PRIOR BELIEF
Prior distribution: Beta(8, 1.5)
Expected prior stiffness efficiency: 0.8421



TASK 2: STRUCTURAL LIKELIHOOD

Measurement model:
Y_k = theta * K_nominal * exp(epsilon_k)
epsilon_k ~ N(0, sigma^2)

Single-measurement likelihood:
L(y_k | theta) = 1 / (y_k * sigma * sqrt(2*pi)) * exp(-(ln(y_k/(theta*K_nominal)))^2/(2*sigma^2))

Joint likelihood:
L(y^(k) | theta) = Product[L(y_i | theta)], i = 1,...,k

TASK 3: NON-CONJUGATE BAYESIAN UPDATE

The Beta prior is not conjugate to the log-normal measurement likelihood.

Therefore, the posterior generally does not have a closed-form standard distribution.

Recursive update:
f(theta | y^(k)) proportional to L(y_k | theta) * f(theta | y^(k-1))

The normalized posterior is:
f(theta | y^(k)) = [L(y_k | theta) f(theta | y^(k-1))] / Z_k

where Z_k is the integral of the numerator over the physical domain.

TASK 4: RUNNING POINT ESTIMATORS

Posterior Mean:
theta_Bayes = integral(theta * posterior(theta) dtheta)

MAP:
theta_MAP = argmax_theta posterior(theta)

Both quantities are evaluated numerically on the bounded theta grid.

TA


TASK 6: SEQUENTIAL STRUCTURAL HEALTH MONITORING RESULTS
Step    Sensor Reading    Bayes Mean        MAP               Variance          Error Mean     
----------------------------------------------------------------------------------------------------
0       Prior             0.8421            0.9331            0.012662          0.1621         
1       41.5020           0.8583            0.8886            0.006297          0.1783         
2       30.2911           0.7622            0.7539            0.005672          0.0822         
3       32.4246           0.7256            0.7187            0.003774          0.0456         
4       29.3044           0.6890            0.6835            0.002616          0.0090         
5       28.9549           0.6659            0.6617            0.001971          0.0141         
6       27.4003           0.6449            0.6414            0.001546          0.0351         
7       37.0039           0.6576            0.6543            0.001379    

# Q. Gaussian Mixture Clustering as Conditional Updating

1

In [4]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

K = 3
n_samples = 1500

phi = np.array([0.30, 0.40, 0.30])

mu = np.array([
    [-2.0, -1.5],
    [2.0, 0.0],
    [0.0, 2.5]
])

Sigma = np.array([
    [[0.8, 0.2],
     [0.2, 0.6]],

    [[0.7, -0.1],
     [-0.1, 0.9]],

    [[0.6, 0.0],
     [0.0, 0.7]]
])

cluster_labels = np.random.choice(
    K,
    size=n_samples,
    p=phi
)

X = np.zeros((n_samples, 2))

for k in range(K):
    indices = np.where(cluster_labels == k)[0]

    X[indices] = np.random.multivariate_normal(
        mean=mu[k],
        cov=Sigma[k],
        size=len(indices)
    )

def gaussian_pdf_2d(X, mean, covariance):
    d = X.shape[1]

    covariance_inv = np.linalg.inv(covariance)
    covariance_det = np.linalg.det(covariance)

    diff = X - mean

    exponent = -0.5 * np.sum(
        (diff @ covariance_inv) * diff,
        axis=1
    )

    coefficient = 1 / (
        (2 * np.pi) ** (d / 2)
        * np.sqrt(covariance_det)
    )

    return coefficient * np.exp(exponent)

x1 = np.linspace(-5, 5, 120)
x2 = np.linspace(-5, 5, 120)

X1, X2 = np.meshgrid(x1, x2)

grid_points = np.column_stack([
    X1.ravel(),
    X2.ravel()
])

mixture_density = np.zeros(len(grid_points))

component_densities = []

for k in range(K):
    density_k = gaussian_pdf_2d(
        grid_points,
        mu[k],
        Sigma[k]
    )

    component_densities.append(density_k)

    mixture_density += phi[k] * density_k

mixture_density = mixture_density.reshape(X1.shape)

fig = go.Figure()

fig.add_trace(
    go.Surface(
        x=X1,
        y=X2,
        z=mixture_density,
        colorscale="Viridis",
        colorbar=dict(
            title="Density"
        ),
        opacity=0.9,
        name="GMM Marginal Density"
    )
)

fig.update_layout(
    title={
        "text": "Marginal Density of a Gaussian Mixture Model",
        "x": 0.5,
        "xanchor": "center"
    },
    scene=dict(
        xaxis_title="X₁",
        yaxis_title="X₂",
        zaxis_title="p(x)"
    ),
    template="plotly_white",
    width=950,
    height=750
)

fig.show()

2

In [5]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

K = 3

phi = np.array([0.30, 0.40, 0.30])

mu = np.array([
    [-2.0, -1.5],
    [2.0, 0.0],
    [0.0, 2.5]
])

Sigma = np.array([
    [[0.8, 0.2],
     [0.2, 0.6]],

    [[0.7, -0.1],
     [-0.1, 0.9]],

    [[0.6, 0.0],
     [0.0, 0.7]]
])

def gaussian_pdf(x, mean, covariance):
    d = len(mean)

    diff = x - mean
    covariance_inv = np.linalg.inv(covariance)
    covariance_det = np.linalg.det(covariance)

    coefficient = 1 / (
        (2 * np.pi) ** (d / 2)
        * np.sqrt(covariance_det)
    )

    exponent = -0.5 * (
        diff @ covariance_inv @ diff
    )

    return coefficient * np.exp(exponent)

x_i = np.array([0.5, 0.5])

component_likelihoods = np.zeros(K)

for k in range(K):
    component_likelihoods[k] = gaussian_pdf(
        x_i,
        mu[k],
        Sigma[k]
    )

weighted_likelihoods = phi * component_likelihoods

marginal_density = np.sum(weighted_likelihoods)

responsibilities = weighted_likelihoods / marginal_density

print("Observation x_i:")
print(x_i)

print("\nMixture Weights:")
print(phi)

print("\nGaussian Likelihoods:")
print(component_likelihoods)

print("\nWeighted Likelihoods:")
print(weighted_likelihoods)

print("\nMarginal Density p(x_i):")
print(marginal_density)

print("\nResponsibilities γ_ik:")
for k in range(K):
    print(
        f"Cluster {k + 1}: "
        f"{responsibilities[k]:.4f}"
    )

print("\nSum of Responsibilities:")
print(np.sum(responsibilities))

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=[f"Cluster {k + 1}" for k in range(K)],
        y=responsibilities,
        text=[f"{r:.3f}" for r in responsibilities],
        textposition="outside",
        name="Posterior Responsibility"
    )
)

fig.update_layout(
    title={
        "text": "Posterior Cluster Responsibilities for Observation xᵢ",
        "x": 0.5,
        "xanchor": "center"
    },
    xaxis_title="Cluster",
    yaxis_title="Posterior Probability γᵢₖ",
    yaxis=dict(range=[0, 1.1]),
    template="plotly_white",
    width=850,
    height=600
)

fig.show()

Observation x_i:
[0.5 0.5]

Mixture Weights:
[0.3 0.4 0.3]

Gaussian Likelihoods:
[0.00086534 0.03869345 0.01145186]

Weighted Likelihoods:
[0.0002596  0.01547738 0.00343556]

Marginal Density p(x_i):
0.019172540383912796

Responsibilities γ_ik:
Cluster 1: 0.0135
Cluster 2: 0.8073
Cluster 3: 0.1792

Sum of Responsibilities:
0.9999999999999999


3

In [6]:
import numpy as np
from scipy.stats import multivariate_normal
import plotly.graph_objects as go

np.random.seed(42)

K = 3

phi = np.array([0.3, 0.4, 0.3])

mu = np.array([
    [0.0, 0.0],
    [3.0, 3.0],
    [-3.0, 3.0]
])

Sigma = np.array([
    [[1.0, 0.2], [0.2, 1.0]],
    [[1.0, 0.0], [0.0, 1.0]],
    [[1.0, -0.2], [-0.2, 1.0]]
])

X = np.array([
    [0.2, 0.4],
    [2.5, 2.8],
    [-2.7, 2.5],
    [0.8, 0.5],
    [1.5, 1.8]
])

responsibilities = []

for x in X:
    component_probabilities = np.array([
        phi[k] * multivariate_normal.pdf(
            x,
            mean=mu[k],
            cov=Sigma[k]
        )
        for k in range(K)
    ])

    gamma = component_probabilities / np.sum(component_probabilities)
    responsibilities.append(gamma)

responsibilities = np.array(responsibilities)

conditional_expectation = responsibilities.copy()

print("Conditional Expectation E[Z_i | X_i = x_i]")
print()

for i in range(len(X)):
    print(f"Observation {i + 1}: {X[i]}")
    print(
        "E[Z_i | X_i = x_i] =",
        np.round(conditional_expectation[i], 4)
    )
    print(
        "Sum of probabilities =",
        np.round(np.sum(conditional_expectation[i]), 4)
    )
    print()

fig = go.Figure()

for k in range(K):
    fig.add_trace(
        go.Bar(
            x=[f"Observation {i + 1}" for i in range(len(X))],
            y=conditional_expectation[:, k],
            name=f"Cluster {k + 1}"
        )
    )

fig.update_layout(
    title="Soft Cluster Assignment as Conditional Expectation",
    xaxis_title="Observation",
    yaxis_title="E[Z_ik | X_i = x_i]",
    barmode="stack",
    template="plotly_white"
)

fig.show()

Conditional Expectation E[Z_i | X_i = x_i]

Observation 1: [0.2 0.4]
E[Z_i | X_i = x_i] = [9.982e-01 1.000e-03 9.000e-04]
Sum of probabilities = 1.0

Observation 2: [2.5 2.8]
E[Z_i | X_i = x_i] = [0.0025 0.9975 0.    ]
Sum of probabilities = 1.0

Observation 3: [-2.7  2.5]
E[Z_i | X_i = x_i] = [2.000e-04 0.000e+00 9.998e-01]
Sum of probabilities = 1.0

Observation 4: [0.8 0.5]
E[Z_i | X_i = x_i] = [9.924e-01 7.400e-03 2.000e-04]
Sum of probabilities = 1.0

Observation 5: [1.5 1.8]
E[Z_i | X_i = x_i] = [3.275e-01 6.723e-01 1.000e-04]
Sum of probabilities = 1.0



4

In [7]:
import numpy as np
import plotly.graph_objects as go

hard_assignments = np.argmax(responsibilities, axis=1) + 1

print("Soft Cluster Assignments")
print(responsibilities)

print("\nHard Cluster Assignments")
for i, cluster in enumerate(hard_assignments):
    print(f"Observation {i + 1} -> Cluster {cluster}")

fig = go.Figure()

for k in range(K):
    fig.add_trace(
        go.Bar(
            x=[f"Observation {i + 1}" for i in range(len(X))],
            y=responsibilities[:, k],
            name=f"Cluster {k + 1}"
        )
    )

fig.update_layout(
    title="Soft Cluster Assignments",
    xaxis_title="Observation",
    yaxis_title="Posterior Probability",
    barmode="stack",
    template="plotly_white"
)

fig.show()

fig_hard = go.Figure()

fig_hard.add_trace(
    go.Scatter(
        x=[f"Observation {i + 1}" for i in range(len(X))],
        y=hard_assignments,
        mode="markers+text",
        text=[f"C{c}" for c in hard_assignments],
        textposition="top center",
        marker=dict(size=12),
        name="Hard Assignment"
    )
)

fig_hard.update_layout(
    title="Hard Cluster Assignments",
    xaxis_title="Observation",
    yaxis_title="Assigned Cluster",
    yaxis=dict(
        tickmode="linear",
        dtick=1,
        range=[0.5, K + 0.5]
    ),
    template="plotly_white"
)

fig_hard.show()

Soft Cluster Assignments
[[9.98158192e-01 9.61445736e-04 8.80362468e-04]
 [2.46651482e-03 9.97533329e-01 1.56280121e-07]
 [2.45369452e-04 1.17463157e-07 9.99754513e-01]
 [9.92372408e-01 7.40812860e-03 2.19463748e-04]
 [3.27540264e-01 6.72335228e-01 1.24507539e-04]]

Hard Cluster Assignments
Observation 1 -> Cluster 1
Observation 2 -> Cluster 2
Observation 3 -> Cluster 3
Observation 4 -> Cluster 1
Observation 5 -> Cluster 2


5

In [8]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)

K = 3

mu = np.array([
    [0, 0],
    [3, 3],
    [-3, 3]
])

X = np.array([
    [0.2,0.4],
    [2.5,2.8],
    [-2.7,2.5],
    [0.8,0.5],
    [1.5,1.8]
])

responsibilities = np.array([
    [0.85,0.10,0.05],
    [0.05,0.90,0.05],
    [0.05,0.10,0.85],
    [0.70,0.20,0.10],
    [0.30,0.60,0.10]
])


fig = go.Figure()


for k in range(K):

    fig.add_trace(
        go.Scatter(
            x=X[:,0],
            y=X[:,1],
            mode="markers",
            marker=dict(
                size=10,
                opacity=responsibilities[:,k]
            ),
            name=f"Soft Membership Cluster {k+1}"
        )
    )


fig.add_trace(
    go.Scatter(
        x=mu[:,0],
        y=mu[:,1],
        mode="markers+text",
        text=[f"μ{k+1}" for k in range(K)],
        textposition="top center",
        marker=dict(
            size=18,
            symbol="x"
        ),
        name="Cluster Centers μk"
    )
)


fig.update_layout(
    title="Conditional Expectations in Gaussian Mixture Model",
    xaxis_title="Feature 1",
    yaxis_title="Feature 2",
    template="plotly_white"
)

fig.show()

6

In [9]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import multivariate_normal

np.random.seed(42)

# ============================================================
# Generate Synthetic GMM Data with Known Latent Labels
# ============================================================

n = 150
K = 3

phi = np.array([0.3, 0.4, 0.3])

mu_true = np.array([
    [0, 0],
    [4, 4],
    [-3, 4]
])

Sigma_true = np.array([
    [[1, 0.2],
     [0.2, 1]],

    [[1, -0.3],
     [-0.3, 1]],

    [[1, 0],
     [0, 1]]
])


# Generate cluster labels
labels = np.random.choice(
    K,
    size=n,
    p=phi
)


# Generate observations
X = np.zeros((n,2))

for i in range(n):
    X[i] = np.random.multivariate_normal(
        mu_true[labels[i]],
        Sigma_true[labels[i]]
    )


# ============================================================
# One-Hot Encoding of Latent Variables Z
# ============================================================

Z = np.zeros((n,K))

for i in range(n):
    Z[i, labels[i]] = 1


print("First 10 One-Hot Latent Variables:")
print(Z[:10])


# ============================================================
# Complete Data Likelihood
# ============================================================

complete_log_likelihood = 0


for i in range(n):

    for k in range(K):

        if Z[i,k] == 1:

            probability = (
                phi[k] *
                multivariate_normal.pdf(
                    X[i],
                    mean=mu_true[k],
                    cov=Sigma_true[k]
                )
            )

            complete_log_likelihood += np.log(probability)


print("\nComplete Data Log Likelihood:")
print(complete_log_likelihood)


# ============================================================
# Visualization 1:
# Data Points with Known Cluster Labels
# ============================================================

fig1 = go.Figure()


for k in range(K):

    cluster_points = X[labels == k]

    fig1.add_trace(
        go.Scatter(
            x=cluster_points[:,0],
            y=cluster_points[:,1],
            mode="markers",
            name=f"Cluster {k+1}"
        )
    )


fig1.add_trace(
    go.Scatter(
        x=mu_true[:,0],
        y=mu_true[:,1],
        mode="markers+text",
        text=[
            "μ1",
            "μ2",
            "μ3"
        ],
        textposition="top center",
        marker=dict(
            size=18,
            symbol="x"
        ),
        name="Gaussian Means"
    )
)


fig1.update_layout(
    title="Complete Data Scenario: Known Cluster Labels",
    xaxis_title="Feature 1",
    yaxis_title="Feature 2",
    template="plotly_white"
)


fig1.show()



# ============================================================
# Visualization 2:
# One-Hot Representation of Latent Variables
# ============================================================

fig2 = go.Figure()


for k in range(K):

    fig2.add_trace(
        go.Bar(
            x=[
                f"Point {i+1}"
                for i in range(20)
            ],
            y=Z[:20,k],
            name=f"z{i}"
        )
    )


fig2.update_layout(
    title="One-Hot Latent Variables zᵢₖ",
    xaxis_title="Observation",
    yaxis_title="zᵢₖ Value",
    barmode="stack",
    template="plotly_white"
)


fig2.show()



# ============================================================
# Visualization 3:
# Contribution of Each Cluster to Log Likelihood
# ============================================================

cluster_log_values = np.zeros(K)


for k in range(K):

    for i in range(n):

        if Z[i,k] == 1:

            cluster_log_values[k] += np.log(
                phi[k] *
                multivariate_normal.pdf(
                    X[i],
                    mean=mu_true[k],
                    cov=Sigma_true[k]
                )
            )


fig3 = go.Figure()


fig3.add_trace(
    go.Bar(
        x=[
            "Cluster 1",
            "Cluster 2",
            "Cluster 3"
        ],
        y=cluster_log_values
    )
)


fig3.update_layout(
    title="Complete Data Log-Likelihood Contribution by Cluster",
    xaxis_title="Cluster",
    yaxis_title="Log Likelihood Contribution",
    template="plotly_white"
)


fig3.show()

First 10 One-Hot Latent Variables:
[[0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [0. 0. 1.]]

Complete Data Log Likelihood:
-576.9407332749315


7

In [10]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import multivariate_normal


np.random.seed(42)


# ============================================================
# Generate Synthetic GMM Dataset
# ============================================================

n = 150
K = 3


phi = np.array([
    0.3,
    0.4,
    0.3
])


mu = np.array([
    [0,0],
    [4,4],
    [-3,4]
])


Sigma = np.array([

    [
        [1,0.2],
        [0.2,1]
    ],

    [
        [1,-0.3],
        [-0.3,1]
    ],

    [
        [1,0],
        [0,1]
    ]

])


# Generate data

X = []
true_labels = []


for i in range(n):

    cluster = np.random.choice(
        K,
        p=phi
    )

    point = np.random.multivariate_normal(
        mu[cluster],
        Sigma[cluster]
    )

    X.append(point)
    true_labels.append(cluster)


X = np.array(X)



# ============================================================
# E-STEP
# Calculate Responsibilities γik
# ============================================================


responsibilities = np.zeros(
    (n,K)
)


for i in range(n):

    posterior_values = np.zeros(K)


    for k in range(K):

        posterior_values[k] = (
            phi[k] *
            multivariate_normal.pdf(
                X[i],
                mean=mu[k],
                cov=Sigma[k]
            )
        )


    responsibilities[i] = (
        posterior_values /
        np.sum(posterior_values)
    )


gamma = responsibilities



print("First 10 Responsibility Vectors")
print(np.round(gamma[:10],4))



# ============================================================
# Conditional Expectation of Z
#
# E[Zik | Xi=xi] = γik
# ============================================================

expected_Z = gamma.copy()


print("\nConditional Expectation E[Zi | Xi=xi]")
print(np.round(expected_Z[:10],4))



# ============================================================
# Visualization 1:
# Soft Assignment Probabilities
# ============================================================


fig1 = go.Figure()


for k in range(K):

    fig1.add_trace(
        go.Bar(
            x=[
                f"Point {i+1}"
                for i in range(20)
            ],

            y=gamma[:20,k],

            name=f"Cluster {k+1}"
        )
    )


fig1.update_layout(

    title="E-Step: Responsibilities γik = E[Zik | Xi=xi]",

    xaxis_title="Observation",

    yaxis_title="Posterior Probability",

    barmode="stack",

    template="plotly_white"

)


fig1.show()



# ============================================================
# Visualization 2:
# Data Colored by Maximum Responsibility
# ============================================================


hard_labels = np.argmax(
    gamma,
    axis=1
)


fig2 = go.Figure()


for k in range(K):

    cluster_points = X[
        hard_labels == k
    ]


    fig2.add_trace(

        go.Scatter(

            x=cluster_points[:,0],

            y=cluster_points[:,1],

            mode="markers",

            name=f"Cluster {k+1}"

        )

    )



fig2.update_layout(

    title="Clusters After Conditional Membership Update",

    xaxis_title="Feature 1",

    yaxis_title="Feature 2",

    template="plotly_white"

)


fig2.show()



# ============================================================
# Visualization 3:
# Responsibility Surface Approximation
# ============================================================


x_range = np.linspace(
    X[:,0].min()-1,
    X[:,0].max()+1,
    100
)

y_range = np.linspace(
    X[:,1].min()-1,
    X[:,1].max()+1,
    100
)


xx,yy = np.meshgrid(
    x_range,
    y_range
)


grid_points = np.column_stack(
    [
        xx.ravel(),
        yy.ravel()
    ]
)



grid_gamma = np.zeros(
    (len(grid_points),K)
)



for i,point in enumerate(grid_points):

    values = np.zeros(K)


    for k in range(K):

        values[k] = (
            phi[k] *
            multivariate_normal.pdf(
                point,
                mean=mu[k],
                cov=Sigma[k]
            )
        )


    grid_gamma[i] = (
        values /
        np.sum(values)
    )



dominant_cluster = np.argmax(
    grid_gamma,
    axis=1
)



fig3 = go.Figure()


fig3.add_trace(

    go.Scatter(

        x=grid_points[:,0],

        y=grid_points[:,1],

        mode="markers",

        marker=dict(

            size=3,

            color=dominant_cluster

        ),

        name="Posterior Cluster Regions"

    )

)



fig3.add_trace(

    go.Scatter(

        x=X[:,0],

        y=X[:,1],

        mode="markers",

        marker=dict(

            size=8,

            symbol="circle-open"

        ),

        name="Observations"

    )

)


fig3.update_layout(

    title="Conditional Update Map from E-Step",

    xaxis_title="Feature 1",

    yaxis_title="Feature 2",

    template="plotly_white"

)


fig3.show()

First 10 Responsibility Vectors
[[0.000e+00 1.000e+00 0.000e+00]
 [1.000e+00 0.000e+00 0.000e+00]
 [1.000e+00 0.000e+00 0.000e+00]
 [1.000e+00 0.000e+00 0.000e+00]
 [6.000e-04 9.994e-01 0.000e+00]
 [1.000e+00 0.000e+00 0.000e+00]
 [0.000e+00 1.000e+00 0.000e+00]
 [0.000e+00 1.000e+00 0.000e+00]
 [0.000e+00 1.000e+00 0.000e+00]
 [9.720e-01 0.000e+00 2.800e-02]]

Conditional Expectation E[Zi | Xi=xi]
[[0.000e+00 1.000e+00 0.000e+00]
 [1.000e+00 0.000e+00 0.000e+00]
 [1.000e+00 0.000e+00 0.000e+00]
 [1.000e+00 0.000e+00 0.000e+00]
 [6.000e-04 9.994e-01 0.000e+00]
 [1.000e+00 0.000e+00 0.000e+00]
 [0.000e+00 1.000e+00 0.000e+00]
 [0.000e+00 1.000e+00 0.000e+00]
 [0.000e+00 1.000e+00 0.000e+00]
 [9.720e-01 0.000e+00 2.800e-02]]


8

In [ ]:
Once we have computed the posterior probabilities (responsibilities $\gamma_{ik}$) in the conditional update step (the E-step), the final phase of framing Gaussian Mixture Clustering as an iterative update process is **updating the model parameters** ($\phi_k, \mu_k, \Sigma_k$) to maximize the expected log-likelihood (the M-step).

Here is how the parameters are updated conditionally based on the responsibilities:

1. Effective Number of Points in a Cluster ($N_k$)

First, we compute the total responsibility assigned to cluster $k$ across all $N$ data points. This represents the effective number of observations belonging to cluster $k$:

$$N_k = \sum_{i=1}^{N} \gamma_{ik}$$

---

2. Updating the Mixture Weights ($\phi_k$)

The prior probability or mixing proportion for cluster $k$ is updated as the fraction of the total dataset assigned to that cluster:

$$\phi_k^{\text{new}} = \frac{N_k}{N}$$

---

3. Updating the Cluster Means ($\mu_k$)

The mean vector of each Gaussian component is updated by taking a weighted average of all data points, where the weights are the responsibilities:

$$\mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^{N} \gamma_{ik} x_i$$

---

4. Updating the Covariance Matrices ($\Sigma_k$)

Similarly, the covariance matrix for each component is updated by computing the sample covariance of the data points around the *new* mean $\mu_k^{\text{new}}$, weighted by the responsibilities:

$$\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^{N} \gamma_{ik} (x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T$$

---

**Interpretation as a Global System Update:**

In this framework, the clustering process operates as a two-stage conditional updating engine:

1. **Local Update (Part 2):** Fix the cluster parameters and update our beliefs about which data points belong to which clusters ($\gamma_{ik}$).
2. **Global Update (Part 3):** Fix those point assignments ($\gamma_{ik}$) and update the structural parameters of the clusters to better fit the data.

Let me know if there's a **Part 4** or an implementation step you'd like to look at next!

9

Gaussian Mixture Model (GMM) clustering can be viewed as a repeated process of conditional updating because it iteratively refines its structural assumptions as new data evidence is ingested. At the start of an iteration, the mixture weight $\phi_k$ acts as our baseline **prior probability** of selecting cluster $k$. Upon encountering an observed data point $x_i$, the model evaluates the Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$, which mathematically **measures how compatible** $x_i$ is with that specific cluster's current shape and location. Combining this prior and likelihood via Bayes' Theorem yields the responsibility $\gamma_{ik}$, representing the **posterior probability** that the data point belongs to cluster $k$ given its observed position.

Aggregated across all components, this collection of responsibilities defines the **soft assignment vector** $\mathbb{E}[Z_i \mid X_i = x_i]$, capturing our complete, updated state of uncertainty. Finally, the M-step closes the loop by **updating the cluster parameters** ($\phi_k, \mu_k, \Sigma_k$) using these newly updated posterior membership probabilities as weights to re-center and reshape the components. From this cyclical process, we can conclude that Gaussian mixture clustering is fundamentally a form of **probabilistic clustering based on conditional expectations of latent cluster membership variables**.

10

In [11]:
!pip install kagglehub -q

import kagglehub
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# ==========================================================
# Download Dataset
# ==========================================================

path = kagglehub.dataset_download(
    "arjunbhasin2013/ccdata"
)

print(path)

df = pd.read_csv(path + "/CC GENERAL.csv")

df.head()


# ==========================================================
# GMM Financial Segmenter Class
# ==========================================================

class GMMFinancialSegmenter:

    def __init__(self, n_components=3, random_state=42):

        self.n_components = n_components
        self.random_state = random_state

        self.scaler = StandardScaler()

        self.model = GaussianMixture(
            n_components=n_components,
            covariance_type="full",
            random_state=random_state
        )


    def prepare_data(self, df, features):

        X = df[features].dropna().values

        X_scaled = self.scaler.fit_transform(X)


        X_train, X_test = train_test_split(
            X_scaled,
            test_size=0.2,
            random_state=self.random_state
        )

        return X_train, X_test



    def fit(self, X_train):

        self.model.fit(X_train)

        print("GMM Training Completed")
        print("----------------------")
        print("Converged:",
              self.model.converged_)

        print("Iterations:",
              self.model.n_iter_)



    def evaluate(self,X_test):

        score = self.model.score(X_test)

        print("\nTest Set Performance")
        print("--------------------")
        print("Average Log Likelihood:",
              round(score,4))

        return score



    def density_heatmap(self,X_train,features):

        X_original = self.scaler.inverse_transform(
            X_train
        )


        fig = px.density_heatmap(
            x=X_original[:,0],
            y=X_original[:,1],
            marginal_x="histogram",
            marginal_y="histogram",
            labels={
                "x":features[0],
                "y":features[1]
            },
            title=
            "Training Data Density Heatmap"
        )


        fig.update_layout(
            template="plotly_white"
        )

        fig.show()



    def contour_map(self,X):

        x_min,x_max = (
            X[:,0].min()-0.5,
            X[:,0].max()+0.5
        )

        y_min,y_max = (
            X[:,1].min()-0.5,
            X[:,1].max()+0.5
        )


        xx,yy=np.meshgrid(
            np.linspace(x_min,x_max,200),
            np.linspace(y_min,y_max,200)
        )


        grid=np.c_[xx.ravel(),
                   yy.ravel()]


        probabilities = (
            self.model.predict_proba(grid)
        )


        confidence = (
            probabilities.max(axis=1)
            .reshape(xx.shape)
        )


        grid_original = (
            self.scaler.inverse_transform(grid)
        )


        return (
            grid_original[:,0].reshape(xx.shape),
            grid_original[:,1].reshape(yy.shape),
            confidence
        )



    def assignment_plot(self,X,title):

        xx,yy,confidence = (
            self.contour_map(X)
        )


        X_original = (
            self.scaler.inverse_transform(X)
        )


        labels = self.model.predict(X)


        fig = go.Figure()


        fig.add_trace(
            go.Contour(
                x=xx[0,:],
                y=yy[:,0],
                z=confidence,
                opacity=0.6,
                colorscale="Viridis",
                showscale=True,
                name="Responsibility Confidence"
            )
        )


        for k in range(self.n_components):

            mask = labels==k


            fig.add_trace(
                go.Scatter(
                    x=X_original[mask,0],
                    y=X_original[mask,1],
                    mode="markers",
                    name=f"Cluster {k+1}",
                    marker=dict(size=6)
                )
            )


        fig.update_layout(
            title=title,
            template="plotly_white",
            xaxis_title="PURCHASES",
            yaxis_title="CREDIT_LIMIT"
        )


        fig.show()



# ==========================================================
# Model Execution
# ==========================================================


features=[
    "PURCHASES",
    "CREDIT_LIMIT"
]


segmenter = GMMFinancialSegmenter(
    n_components=3
)


X_train,X_test = (
    segmenter.prepare_data(
        df,
        features
    )
)


segmenter.fit(X_train)


segmenter.evaluate(X_test)


# ==========================================================
# Interactive Visualizations
# ==========================================================


segmenter.density_heatmap(
    X_train,
    features
)


segmenter.assignment_plot(
    X_train,
    "GMM Soft Assignment Map - Training Data"
)


segmenter.assignment_plot(
    X_test,
    "GMM Soft Assignment Map - Test Data"
)

100%|██████████| 340k/340k [00:00<00:00, 51.0MB/s]

Extracting files...
/root/.cache/kagglehub/datasets/arjunbhasin2013/ccdata/versions/1


GMM Training Completed
----------------------
Converged: True
Iterations: 19

Test Set Performance
--------------------
Average Log Likelihood: -1.6465
